# Security report generation
Generates complete and per-finding HTML/JSON reports and refuses unsupported findings.

In [ ]:
from pathlib import Path
from html import escape
import hashlib, json, os
INPUT=Path(os.getenv('REPORT_INPUT_PATH','scan_report_input.json'))
ROOT=Path(os.getenv('REPORT_OUTPUT_ROOT','reports'))
if not INPUT.exists(): raise FileNotFoundError(f'Required input not found: {INPUT.resolve()}')
data=json.loads(INPUT.read_text()); scan=data['scan']; findings=data.get('findings',[])
unsupported=[f.get('id','unknown') for f in findings if not f.get('evidence')]
if unsupported: raise ValueError(f'Report refused findings without evidence: {unsupported}')
out=ROOT/scan['id']; (out/'findings').mkdir(parents=True,exist_ok=True)
print(f'Generating reports for {len(findings)} evidence-backed findings')

In [ ]:
STYLE='body{font:15px/1.6 Helvetica,Arial,sans-serif;color:#242520;max-width:1050px;margin:40px auto;padding:0 24px}h1,h2{font-family:Georgia,serif}section{border-top:1px solid #ddd;padding:24px 0}pre{background:#20221f;color:#f4f3ee;padding:16px;white-space:pre-wrap}'
def shell(title,body): return f'<!doctype html><html><head><meta charset="utf-8"><meta http-equiv="Content-Security-Policy" content="default-src &#39;none&#39;; style-src &#39;unsafe-inline&#39;"><title>{escape(title)}</title><style>{STYLE}</style></head><body>{body}</body></html>'
def finding_html(f):
    evidence=''.join(f'<pre>{escape(e.get("id",""))} · {escape(e.get("source",""))}\nSHA-256 {escape(e.get("hash",""))}\n\n{escape(e.get("redacted_excerpt",e.get("redactedExcerpt","")))}</pre>' for e in f['evidence'])
    return shell(f['id']+' — '+f['title'],f'<h1>{escape(f["title"])}</h1><p>{escape(f["severity"])} · {round(float(f["confidence"])*100)}% confidence</p><section><h2>Detailed description</h2><p>{escape(f["description"])}</p><h2>Technical impact</h2><p>{escape(f["technical_impact"])}</p><h2>Business impact</h2><p>{escape(f["business_impact"])}</p></section><section><h2>Evidence</h2>{evidence}</section><section><h2>Remediation</h2><p>{escape(f["remediation"])}</p><h2>Validation / retest guidance</h2><p>Repeat the original non-destructive request and preserve the secured response as retest evidence.</p></section>')
rows=[]
for f in findings:
    html=finding_html(f); base=out/'findings'/f['id']; base.with_suffix('.html').write_text(html); base.with_suffix('.json').write_text(json.dumps(f,indent=2)+'\n'); rows.append(f'<tr><td>{escape(f["id"])}</td><td>{escape(f["title"])}</td><td>{escape(f["severity"])}</td><td>{round(float(f["confidence"])*100)}%</td></tr>')
body=f'<h1>{escape(scan.get("name","Complete scan report"))}</h1><p>Scan ID {escape(scan["id"])}</p><section><h2>Executive summary</h2><p>{len(findings)} evidence-backed findings were retained.</p></section><section><h2>Findings summary</h2><table>{"".join(rows)}</table></section>'
complete=shell('Complete scan report',body); (out/'scan-report.html').write_text(complete); (out/'scan-report.pdf-ready.html').write_text(complete); (out/'scan-report.json').write_text(json.dumps(data,indent=2)+'\n')
manifest={'scan_id':scan['id'],'findings':len(findings),'scan_report_sha256':hashlib.sha256(complete.encode()).hexdigest()}
json.loads((out/'scan-report.json').read_text()); print(json.dumps(manifest,indent=2)); print(f'Validated output: {out.resolve()}')